# mcp

> The MCP frontend: a stdio router over rustygate gateways

In [ ]:
#| default_exp mcp

The frontend Claude Code launches per conversation. `Router` speaks stdio MCP to the harness and forwards to gateways: rustygate serves the tool surface itself, so the router defines no tools — it fetches the local gateway's `tools/list`, adds a `host` parameter to the kernel-selection tools, and forwards `tools/call` verbatim to the right gateway, ids intact so cancellation maps through. `main` serves a `Router` on stdio and ends every gateway session on the way out — the DELETE that stops each session's autoclose kernels, and the owned child gateway with them.


In [ ]:
#| export
import asyncio
from fastcore.utils import *
from fastcore.script import call_parse, store_true
from mcpmini.core import serve_stdio, jresp, jerr
from clikernel.core import Gateway, default_gateway, resolve, session_defaults
from clikernel import __version__


In [ ]:
from fastcore.test import *
import os, re, tempfile
from mcpmini.core import jreq, jtool
from rustygate.tools import start_gateway

In [ ]:
#| export
HOST_PARAM = {'type': 'string', 'description': 'Gateway to target: a gateways.toml name, or empty for the default local gateway'}
HOSTED = ('list_kernels', 'use_kernel', 'create')

class Router:
    "Forward the harness's stdio MCP to rustygate gateways: one `Gateway` session per host, one current"
    def __init__(self,
        cfgdir=None,  # Config dir for `session_defaults` and `gateways.toml` (the standard one if None)
        quiet=False,  # Keep startup output out of replies?
    ):
        self.cfgdir,self.quiet,self.sessions,self.cur,self.child = cfgdir,quiet,{},'',None

    async def session(self, host=''):
        "The initialized `Gateway` for `host`, made on first use; empty means the default local gateway"
        if host not in self.sessions:
            if host:
                url, token, verify = resolve(host, self.cfgdir)
                self.sessions[host] = await Gateway(url, token, verify).initialize(session_defaults(self.cfgdir, self.quiet, local=False))
            else: self.sessions[host], self.child = await default_gateway(self.cfgdir, self.quiet)
        return self.sessions[host]

    async def aclose(self):
        "End every gateway session — stopping each one's autoclose kernels — then the owned child gateway"
        for s in self.sessions.values(): await s.aclose()
        if self.child: self.child.stop()


The dispatch is the whole protocol story. `initialize` is answered locally so the router can negotiate with the harness while holding its own session gateway-side; `tools/list` is the fetch-and-patch; `tools/call` forwards the harness's message verbatim — same JSON-RPC id, so a later cancellation notification names a request the gateway recognizes. Only `use_kernel` and `create` carrying a `host` move the current gateway:

In [ ]:
#| export
@patch
async def tools(self:Router):
    "The local gateway's tools, with `host` added to the kernel-selection tools"
    ts = await (await self.session()).tools()
    for t in ts:
        if t['name'] in HOSTED: t['inputSchema'].setdefault('properties', {})['host'] = dict(HOST_PARAM)
    return ts

@patch
async def dispatch(self:Router, msg, requester=None):
    "One JSON-RPC message from the harness: initialize and ping answered here, the tool surface forwarded"
    method,id = msg.get('method'), msg.get('id')
    try:
        if method == 'initialize':
            info = (await self.session()).info
            return jresp(id, dict(protocolVersion=msg['params'].get('protocolVersion', '2025-06-18'), capabilities=dict(tools={}),
                serverInfo=dict(name='clikernel', version=__version__), instructions=info.get('instructions')))
        if id is None:
            if method == 'notifications/cancelled': await (await self.session(self.cur)).tr.send(msg)
            return None
        if method == 'ping': return jresp(id, {})
        if method == 'tools/list': return jresp(id, dict(tools=await self.tools()))
        if method == 'tools/call':
            args = msg['params'].setdefault('arguments', {})
            has_host = 'host' in args
            host = args.pop('host', '') or ''
            s = await self.session(host if has_host else self.cur)
            if has_host and msg['params']['name'] in ('use_kernel', 'create'): self.cur = host
            return await s.tr.send(msg)
        return jerr(id, -32601, f'method not found: {method}')
    except Exception as e: return None if id is None else jerr(id, -32603, str(e))

Directly at the dispatch level, against a live rustygate on its own port. The router answers `initialize` itself — rustygate's instructions forwarded, since the gateway owns the contract prose — and everything else it can, it forwards:


In [ ]:
g = start_gateway()
os.environ['CLIKERNEL_HOST'] = g.url
cfgd = Path(tempfile.mkdtemp())
(cfgd/'startup.py').write_text('base = 42; print("ready")')
router = Router(cfgd)
init = await router.dispatch(jreq('initialize', 1, protocolVersion='2025-11-25', capabilities={}, clientInfo=dict(name='demo', version='0')))
test_eq(init['result']['serverInfo']['name'], 'clikernel')
init['result']['instructions']


"py runs Python/IPython; lua runs bundled Luau. Either starts its language's kernel when none is current, stopped at session end. There is one current kernel; a language mismatch errors without switching. create accepts language=python or luau; an omitted language reuses an existing binding or defaults to Python for a new one. A dlgname execution override applies to one call only. Python startup/inspectors never run in Luau."

In [ ]:
def txt(r): return ''.join(c.get('text','') for c in r['result']['content'] if c['type'] == 'text')
listed = (await router.dispatch(jreq('tools/list', 2)))['result']['tools']
test_eq([t['name'] for t in listed], ['list_kernels', 'create', 'use_kernel', 'delete_kernel', 'py', 'lua', 'restart', 'interrupt'])
byname = {t['name']: t for t in listed}
assert all('host' in byname[n]['inputSchema']['properties'] for n in HOSTED)
assert 'host' not in byname['py']['inputSchema']['properties']
r = await router.dispatch(jtool('py', 3, code='base'))
assert 'created kernel' in txt(r) and 'ready' in txt(r) and txt(r).endswith('42')
txt(r)


'created kernel bed3b2bc607b466b8af7d57329ae1ef3 language=python\nready\n42'

`host` reaches another gateway by its `gateways.toml` name. `use_kernel` and `create` carrying a host also move the session's current gateway, so later `py` calls land there; `list_kernels` with a host only looks.

In [ ]:
g2 = start_gateway()
(cfgd/'gateways.toml').write_text(f'[gateways.alt]\nurl = "{g2.url}"\n')
r = await router.dispatch(jtool('create', 4, dlgname='far.ipynb', host='alt'))
assert 'created kernel' in txt(r) and 'ready' in txt(r)
await router.dispatch(jtool('py', 5, code='marker = 7'))
test_eq(txt(await router.dispatch(jtool('py', 6, code='marker'))), '7')

Selecting home again is `use_kernel` with an explicit empty `host`. State says which kernel you are on: the far kernel holds `marker`, the local auto kernel holds `base` — and the far gateway shows its binding:

In [ ]:
kid = re.search(r'^(\w+) ', txt(await router.dispatch(jtool('list_kernels', 7, host=''))), re.M).group(1)
await router.dispatch(jtool('use_kernel', 8, kernel=kid, host=''))
assert 'NameError' in txt(await router.dispatch(jtool('py', 9, code='marker')))
test_eq(txt(await router.dispatch(jtool('py', 10, code='base'))), '42')
txt(await router.dispatch(jtool('list_kernels', 11, host='alt')))

'1a2105b534d14ac3a09b93f6181fb712  alive  language=python  connections=1  dlgname=far.ipynb  <- current'

Images through the same forwarded path: a kernel renders a PIL image, and rustygate's reply carries the text representation and an MCP image block side by side — mime preferences follow `fastcore.nbio` (`IMG_MIMES` order picks jpeg from PIL's png-and-jpeg repr). The router forwards content untouched, so what the harness shows is exactly what the gateway rendered:


In [ ]:
r = await router.dispatch(jtool('py', 12, code='from PIL import Image\nImage.new("RGB", (300, 200), "red")'))
blocks = r['result']['content']
test_eq([b['type'] for b in blocks], ['text', 'image'])
test_eq(blocks[1]['mimeType'], 'image/jpeg')
{k: (v[:20] + '…' if k == 'data' else v) for k,v in blocks[1].items()}


{'type': 'image', 'data': '/9j/4AAQSkZJRgABAQAA…', 'mimeType': 'image/jpeg'}

`create(language='luau')` selects a Luau kernel on the named gateway. Python startup does not run there. Sending Python to that kernel fails without switching kernels or changing its state:


In [ ]:
r = await router.dispatch(jtool('create', 20, dlgname='native.ipynb', language='luau', host='alt'))
assert not r['result']['isError'] and 'language=luau' in txt(r)
test_eq(txt(await router.dispatch(jtool('lua', 21, code='saved=42; return saved, base == nil'))), '42\ttrue')
wrong = await router.dispatch(jtool('py', 22, code='saved=0'))
assert wrong['result']['isError'] and 'not python' in txt(wrong)
test_eq(txt(await router.dispatch(jtool('lua', 23, code='saved'))), '42')
txt(wrong)

'kernel 5177f24e169f498886a0f70a91db0979 uses luau, not python; select a matching kernel with use_kernel or create'

`restart` clears the Luau kernel's state without running Python startup. Selecting the original Python kernel restores access to its existing state:

In [ ]:
await router.dispatch(jtool('restart', 24))
test_eq(txt(await router.dispatch(jtool('lua', 25, code='return saved, base'))), 'nil\tnil')
await router.dispatch(jtool('use_kernel', 26, kernel=kid, host=''))
test_eq(txt(await router.dispatch(jtool('py', 27, code='base'))), '42')
txt(await router.dispatch(jtool('list_kernels', 28, host='alt')))

'1a2105b534d14ac3a09b93f6181fb712  alive  language=python  connections=0  dlgname=far.ipynb\n5177f24e169f498886a0f70a91db0979  alive  language=luau  connections=1  dlgname=native.ipynb  <- current'

`main` serves one `Router` on stdio. Sessions to gateways are made on first use — `initialize` reaches the local gateway (starting an owned child when none runs), so the tool list is ready before the harness asks. On the way out, every session ends and an owned gateway stops with everything in it. `--quiet` keeps startup output out of replies.


In [ ]:
#| export
@call_parse
def main(
    quiet:store_true=False,  # Keep startup output out of replies
):
    "The `clikernel-mcp` console script: the router on stdio"
    async def _main():
        router = Router(quiet=quiet)
        try: await serve_stdio(router)
        finally: await router.aclose()
    asyncio.run(_main())


The installed `clikernel-mcp` process exposes the same tools over stdio. This client runs Python, then selects a Luau kernel and runs the same calculation:

In [ ]:
import shutil
from mcpmini.core import MCPClient


In [ ]:
cmd = shutil.which('clikernel-mcp')
assert cmd, 'clikernel-mcp script not installed: run `uv sync`'
env = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(Path(tempfile.mkdtemp()))}
async with MCPClient.stdio([cmd], env=env) as m:
    assert 'py' in dir(m.tools) and 'lua' in dir(m.tools) and 'bash' not in dir(m.tools)
    res = await m.tools.py(code='6*7')
    await m.tools.create(dlgname='stdio-native.ipynb', language='luau')
    native_res = await m.tools.lua(code='6*7')
assert res.endswith('42') and native_res == '42'
native_res


'42'

Ending the router is the whole cleanup story: every gateway session gets its DELETE, so each session's autoclose kernels die — near and far — and merely selected kernels survive. Nothing here owned a child gateway, since the scratch gateways were already running:

In [ ]:
await router.aclose()
assert router.child is None
chk = Router(cfgd)
near,far = [txt(await chk.dispatch(jtool('list_kernels', i, host=h))) for i,h in ((1, ''), (2, 'alt'))]
await chk.aclose()
test_eq((near, far), ('no kernels', 'no kernels'))
near,far


('no kernels', 'no kernels')

## A live client

The proof that matters for deployment: Claude Code's real MCP client, driven headless by the Agent SDK, calling the forwarded tool surface — a bare `py`, auto-starting its kernel — against a live gateway. `#| eval: false`: it spends tokens; rerun manually when the tools or Claude Code move.


In [ ]:
#| eval: false
import logging, random
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage

In [ ]:
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_dir = Path(tempfile.mkdtemp())
lenv = {k:str(v) for k,v in (os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(live_dir)}).items()}
opts = ClaudeAgentOptions(mcp_servers=dict(ck=dict(type='stdio', command=cmd, env=lenv)), cwd=str(live_dir),
    allowed_tools=['mcp__ck__py'], max_turns=6)
prompt = 'Using the ck MCP py tool, compute 17*19 in the kernel. Reply with just the number.'
msgs = [m async for m in query(prompt=prompt, options=opts)]
tus = {b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if isinstance(b, ToolUseBlock)}
res = first(m.result for m in msgs if isinstance(m, ResultMessage))
assert 'mcp__ck__py' in tus
assert '323' in res
res


'323'

And the image path with real Claude: the test host picks a color the model is never told, the kernel renders it as a plain image, and the model answers from the image block alone — proof the block survives dispatch, stdio framing, and the host's own client, and lands in the model's vision context.

In [ ]:
#| eval: false
color = random.choice(['red', 'green', 'blue', 'yellow', 'purple', 'orange'])
(live_dir/'color.txt').write_text(color)
code = "from PIL import Image\nImage.new('RGB', (200,200), open('color.txt').read().strip())"
prompt = ('Using the ck MCP tools: run exactly this code with the py tool (do not run anything else, and do not read color.txt any other way):\n\n'
    f'{code}\n\nThe py result includes an image. Reply with just the color of that image.')
msgs2 = [m async for m in query(prompt=prompt, options=opts)]
res2 = first(m.result for m in msgs2 if isinstance(m, ResultMessage))
assert color in res2.lower()
color, res2

('purple', 'Purple.')

In [ ]:
#|hide
g.stop()
g2.stop()
del os.environ['CLIKERNEL_HOST']


In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()